# 01 – Data Cleaning: Lending Club Loans

**Goal:** Turn the raw Lending Club file (2.26M loans, 151 columns) into an analysis-ready table of completed loans, then load it into PostgreSQL.

**Output:** `data/processed/loans_clean.parquet` and the `loans` table in PostgreSQL (1,347,698 rows, 30 columns).

## 1. Setup

In [1]:
from pathlib import Path
import os
import gc

import pandas as pd
import numpy as np

from dotenv import load_dotenv
from sqlalchemy import create_engine, text


pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

RAW_DIR = Path("../data/raw")
ACCEPTED_FILE = RAW_DIR / "accepted_2007_to_2018Q4.csv"

print(ACCEPTED_FILE.exists())

True


## 2. Load raw data
The file is read directly from the compressed `.csv`. A 5-row preview is used first to check the file and its structure before loading everything.

In [2]:
df = pd.read_csv(ACCEPTED_FILE, low_memory=False)
print(df.shape)
df.head()

(2260701, 151)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,"3,600.00","3,600.00","3,600.00",36 months,13.99,123.03,C,C4,leadman,10+ years,MORTGAGE,"55,000.00",Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,debt_consolidation,Debt consolidation,190xx,PA,5.91,0.00,Aug-2003,675.00,679.00,1.00,30.00,NaN,7.00,0.00,"2,765.00",29.70,13.00,w,0.00,0.00,"4,421.72","4,421.72","3,600.00",821.72,0.00,0.00,0.00,Jan-2019,122.67,NaN,Mar-2019,564.00,560.00,0.00,30.00,1.00,Individual,NaN,NaN,NaN,0.00,722.00,"144,904.00",2.00,2.00,0.00,1.00,21.00,"4,981.00",36.00,3.00,3.00,722.00,34.00,"9,300.00",3.00,1.00,4.00,4.00,"20,701.00","1,506.00",37.20,0.00,0.00,148.00,128.00,3.00,3.00,1.00,4.00,69.00,4.00,69.00,2.00,2.00,4.00,2.00,5.00,3.00,4.00,9.00,4.00,7.00,0.00,0.00,0.00,3.00,76.90,0.00,0.00,0.00,"178,050.00","7,746.00","2,400.00","13,734.00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,"24,700.00","24,700.00","24,700.00",36 months,11.99,820.28,C,C1,Engineer,10+ years,MORTGAGE,"65,000.00",Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,small_business,Business,577xx,SD,16.06,1.00,Dec-1999,715.00,719.00,4.00,6.00,NaN,22.00,0.00,"21,470.00",19.20,38.00,w,0.00,0.00,"25,679.66","25,679.66","24,700.00",979.66,0.00,0.00,0.00,Jun-2016,926.35,NaN,Mar-2019,699.00,695.00,0.00,NaN,1.00,Individual,NaN,NaN,NaN,0.00,0.00,"204,396.00",1.00,1.00,0.00,1.00,19.00,"18,005.00",73.00,2.00,3.00,"6,472.00",29.00,"111,800.00",0.00,0.00,6.00,4.00,"9,7

In [3]:
df.columns.tolist()

['id',
 'member_id',
 'loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'loan_status',
 'pymnt_plan',
 'url',
 'desc',
 'purpose',
 'title',
 'zip_code',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'mths_since_last_record',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'next_pymnt_d',
 'last_credit_pull_d',
 'last_fico_range_high',
 'last_fico_range_low',
 'collections_12_mths_ex_med',
 'mths_since_last_major_derog',
 'policy_code',
 'application_type',
 'annual_inc_joint',
 '

## 3. Data overview
- 2,260,701 rows, 151 columns, ~3.3 GB in memory
- No duplicate loan ids and no duplicate rows

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Columns: 151 entries, id to settlement_term
dtypes: float64(113), str(38)
memory usage: 3.1 GB


In [5]:
df.describe()

,member_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,delinq_2yrs,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_amnt,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,annual_inc_joint,dti_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,deferral_term,hardship_amount,hardship_length,hardship_dpd,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,settlement_amount,settlement_percentage,settlement_term
count,0.00,"2,260,668.00","2,260,668.00","2,260,668.00","2,260,668.00","2,260,668.00","2,260,664.00","2,258,957.00","2,260,639.00","2,260,668.00","2,260,668.00","2,260,638.00","1,102,166.00","359,156.00","2,260,639.00","2,260,639.00","2,260,668.00","2,258,866.00","2,260,639.00","2,260,668.00","2,260,668.00","2,260,668.00","2,260,668.00","2,260,668.00","2,260,668.00","2,260,668.00","2,260,668.00","2,260,668.00","2,260,668.00","2,260,668.00","2,260,668.00","2,260,523.00","580,775.00","2,260,668.00","120,710.00","120,706.00","2,260,639.00","2,190,392.00","2,190,392.00","1,394,538.00","1,394,539.00","1,394,539.00","1,394,539.00","1,350,744.00","1,394,539.00","1,191,818.00","1,394,539.00","1,394,539.00","1,394,539.00","1,394,320.00","2,190,392.00","1,394,539.00","1,394,538.00","1,394,538.00","2,210,638.00","2,190,322.00","2,185,733.00","2,184,597.00","2,260,523.00","2,260,639.00","2,121,597.00","2,190,391.00","2,190,391.00","2,190,392.00","2,210,638.00","2,187,256.00","519,701.00","1,965,233.00","740,359.00","2,190,392.00","2,190,392.00","2,190,392.00","2,202,078.00","2,190,392.00","2,190,392.00","2,190,392.00","2,190,391.00","2,190,392.00","2,202,078.00","2,107,011.00","2,190,392.00","2,190,392.00","2,190,392.00","2,190,237.00","2,185,289.00","2,259,303.00","2,260,563.00","2,190,392.00","2,210,638.00","2,210,638.00","2,190,392.00","108,020.00","108,021.00","108,021.00","108,021.00","108,021.00","108,021.00","106,184.00","108,021.00","108,021.00","108,021.00","108,021.00","35,942.00","10,917.00","10,917.00","10,917.00","10,917.00","8,651.00","10,917.00","10,917.00","34,246.00","34,246.00","34,246.00"
mean,NaN,"15,046.93","15,041.66","15,023.44",13.09,445.81,"77,992.43",18.82,0.31,698.59,702.59,0.58,34.54,72.31,11.61,0.20,"16,658.46",50.34,24.16,"4,206.89","4,205.97","12,082.56","12,064.39","9,505.77","2,431.39",1.52,143.88,23.98,"3,429.35",687.66,675.54,0.02,44.16,1.00,"123,624.64",19.25,0.00,232.73,"142,492.20",0.93,2.78,0.68,1.56,21.22,"35,506.65",69.14,1.29,2.75,"5,806.39",57.03,"34,573.94",1.01,1.48,2.04,4.52,"13,547.80","11,394.26",57.90,0.01,12.37,125.74,181.49,14.02,8.30,1.56,24.84,39.3

In [6]:
print("Rows:              ", len(df))
print("Duplicate ids:     ", df["id"].duplicated().sum())
print("Duplicate rows:    ", df.duplicated().sum())

Rows:               2260701
Duplicate ids:      0
Duplicate rows:     0


In [7]:
print(f"{df.memory_usage(deep=True).sum() / 1e9:.1f} GB")

3.3 GB


## 4. Missing values (all columns)
- 52 columns have (almost) no missing values
- 44 columns are more than 50% empty – mostly joint-application (`sec_app_*`), hardship and settlement fields that apply to very few loans
- 33 rows at the end of the file are summary text, not loans – they are removed by the status filter later

In [8]:
missing_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
missing_pct.head(40)

member_id                                    100.00
orig_projected_additional_accrued_interest    99.60
hardship_reason                               99.50
hardship_payoff_balance_amount                99.50
hardship_last_payment_amount                  99.50
payment_plan_start_date                       99.50
hardship_type                                 99.50
hardship_status                               99.50
hardship_start_date                           99.50
deferral_term                                 99.50
hardship_amount                               99.50
hardship_dpd                                  99.50
hardship_loan_status                          99.50
hardship_length                               99.50
hardship_end_date                             99.50
settlement_status                             98.50
debt_settlement_flag_date                     98.50
settlement_term                               98.50
settlement_percentage                         98.50
settlement_d

In [9]:
print("Columns with no missing values:   ", (missing_pct == 0).sum())
print("Columns with > 50% missing values:", (missing_pct > 50).sum())

Columns with no missing values:    52
Columns with > 50% missing values: 44


In [10]:
df.isna().sum().sort_values().head(10)

id                  0
loan_amnt          33
funded_amnt        33
funded_amnt_inv    33
int_rate           33
term               33
installment        33
grade              33
issue_d            33
sub_grade          33
dtype: int64

## 5. Column selection
Only information **known at origination** is kept, plus the outcome columns needed for loss calculations.

| Group | Columns |
| --- | --- |
| Loan | `loan_amnt`, `term`, `int_rate`, `installment`, `grade`, `sub_grade`, `purpose`, `issue_d` |
| Borrower | `emp_length`, `home_ownership`, `annual_inc`, `verification_status`, `addr_state` |
| Credit history | `dti`, `fico_range_low`, `fico_range_high`, `revol_util`, `open_acc`, `delinq_2yrs`, `inq_last_6mths` |
| Outcome | `loan_status`, `total_pymnt`, `recoveries` |

**Data leakage:** outcome columns are known only after the loan is issued. They are used for loss analysis but must never be model inputs.

In [11]:
SELECTED_COLUMNS = [
    # identifier
    "id",
    # loan
    "loan_amnt", "term", "int_rate", "installment",
    "grade", "sub_grade", "purpose", "issue_d",
    # borrower
    "emp_length", "home_ownership", "annual_inc",
    "verification_status", "addr_state",
    # credit history
    "dti", "fico_range_low", "fico_range_high", "revol_util",
    "open_acc", "delinq_2yrs", "inq_last_6mths",
    # outcome (post-origination: never use as model inputs)
    "loan_status", "total_pymnt", "recoveries",
]

loans = df[SELECTED_COLUMNS].copy()
print(loans.shape)
print(f"{loans.memory_usage(deep=True).sum() / 1e9:.2f} GB")

(2260701, 24)
0.62 GB


In [12]:
loans.head()

,id,loan_amnt,term,int_rate,installment,grade,sub_grade,purpose,issue_d,emp_length,home_ownership,annual_inc,verification_status,addr_state,dti,fico_range_low,fico_range_high,revol_util,open_acc,delinq_2yrs,inq_last_6mths,loan_status,total_pymnt,recoveries
0,68407277,"3,600.00",36 months,13.99,123.03,C,C4,debt_consolidation,Dec-2015,10+ years,MORTGAGE,"55,000.00",Not Verified,PA,5.91,675.00,679.00,29.70,7.00,0.00,1.00,Fully Paid,"4,421.72",0.00
1,68355089,"24,700.00",36 months,11.99,820.28,C,C1,small_business,Dec-2015,10+ years,MORTGAGE,"65,000.00",Not Verified,SD,16.06,715.00,719.00,19.20,22.00,1.00,4.00,Fully Paid,"25,679.66",0.00
2,68341763,"20,000.00",60 months,10.78,432.66,B,B4,home_improvement,Dec-2015,10+ years,MORTGAGE,"63,000.00",Not Verified,IL,10.78,695.00,699.00,56.20,6.00,0.00,0.00,Fully Paid,"22,705.92",0.00
3,66310712,"35,000.00",60 months,14.85,829.90,C,C5,debt_consolidation,Dec-2015,10+ years,MORTGAGE,"110,000.00",Source Verified,NJ,17.06,785.00,789.00,11.60,13.00,0.00,0.00,Current,"31,464.01",0.00
4,68476807,"10,400.00",60 months,22.45,289.91,F,F1,major_purchase,Dec-2015,3 years,MORTGAGE,"104,433.00",Source Verified,PA,25.37,695.00,699.00,64.50,12.00,1.00,3.00,Fully Paid,"11,740.50",0.00


In [13]:
loans.isna().sum().sort_values(ascending=False)

emp_length             146940
revol_util               1835
dti                      1744
inq_last_6mths             63
delinq_2yrs                62
open_acc                   62
annual_inc                 37
loan_amnt                  33
fico_range_low             33
purpose                    33
grade                      33
sub_grade                  33
installment                33
int_rate                   33
term                       33
issue_d                    33
fico_range_high            33
addr_state                 33
verification_status        33
home_ownership             33
total_pymnt                33
loan_status                33
recoveries                 33
id                          0
dtype: int64

In [14]:
loans.info()

<class 'pandas.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 24 columns):
 #   Column               Dtype  
---  ------               -----  
 0   id                   str    
 1   loan_amnt            float64
 2   term                 str    
 3   int_rate             float64
 4   installment          float64
 5   grade                str    
 6   sub_grade            str    
 7   purpose              str    
 8   issue_d              str    
 9   emp_length           str    
 10  home_ownership       str    
 11  annual_inc           float64
 12  verification_status  str    
 13  addr_state           str    
 14  dti                  float64
 15  fico_range_low       float64
 16  fico_range_high      float64
 17  revol_util           float64
 18  open_acc             float64
 19  delinq_2yrs          float64
 20  inq_last_6mths       float64
 21  loan_status          str    
 22  total_pymnt          float64
 23  recoveries           float64
dtypes: float6

In [15]:
loans.describe().T

,count,mean,std,min,25%,50%,75%,max
loan_amnt,"2,260,668.00","15,046.93","9,190.25",500.00,"8,000.00","12,900.00","20,000.00","40,000.00"
int_rate,"2,260,668.00",13.09,4.83,5.31,9.49,12.62,15.99,30.99
installment,"2,260,668.00",445.81,267.17,4.93,251.65,377.99,593.32,"1,719.83"
annual_inc,"2,260,664.00","77,992.43","112,696.20",0.00,"46,000.00","65,000.00","93,000.00","110,000,000.00"
dti,"2,258,957.00",18.82,14.18,-1.00,11.89,17.84,24.49,999.00
fico_range_low,"2,260,668.00",698.59,33.01,610.00,675.00,690.00,715.00,845.00
fico_range_high,"2,260,668.00",702.59,33.01,614.00,679.00,694.00,719.00,850.00
revol_util,"2,258,866.00",50.34,24.71,0.00,31.50,50.30,69.40,892.30
open_acc,"2,260,639.00",11.61,5.64,0.00,8.00,11.00,14.00,101.00
delinq_2yrs,"2,260,639.00",0.31,0.87,0.00,0.00,0.00,0.00,58.00


## 6. Target variable
Only loans with a final outcome are kept. Current and late loans are excluded because their outcome is still unknown.

| loan_status | is_default |
| --- | --- |
| Fully Paid, Does not meet the credit policy (Fully Paid) | 0 |
| Charged Off, Default, Does not meet the credit policy (Charged Off) | 1 |
| Current, In Grace Period, Late (16–30), Late (31–120) | excluded |

Result: **1,348,099 completed loans, default rate 19.98%**.

In [16]:
df["loan_status"].value_counts(dropna=False)

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
NaN                                                         33
Name: count, dtype: int64

In [17]:
del df
gc.collect()

0

In [18]:
STATUS_TO_DEFAULT = {
    "Fully Paid": 0,
    "Does not meet the credit policy. Status:Fully Paid": 0,
    "Charged Off": 1,
    "Does not meet the credit policy. Status:Charged Off": 1,
    "Default": 1,
}

loans = loans[loans["loan_status"].isin(STATUS_TO_DEFAULT.keys())].copy()
loans["is_default"] = loans["loan_status"].map(STATUS_TO_DEFAULT).astype("int8")

print(loans.shape)
print(f"Default rate: {loans['is_default'].mean():.2%}")

(1348099, 25)
Default rate: 19.98%


In [19]:
loans.isna().sum().sort_values(ascending=False).head(8)

emp_length        78550
revol_util          897
dti                 374
inq_last_6mths       30
delinq_2yrs          29
open_acc             29
annual_inc            4
loan_amnt             0
dtype: int64

In [20]:
loans.dtypes

id                         str
loan_amnt              float64
term                       str
int_rate               float64
installment            float64
grade                      str
sub_grade                  str
purpose                    str
issue_d                    str
emp_length                 str
home_ownership             str
annual_inc             float64
verification_status        str
addr_state                 str
dti                    float64
fico_range_low         float64
fico_range_high        float64
revol_util             float64
open_acc               float64
delinq_2yrs            float64
inq_last_6mths         float64
loan_status                str
total_pymnt            float64
recoveries             float64
is_default                int8
dtype: object

In [21]:
for column in ["term", "emp_length", "int_rate", "revol_util", "issue_d", "home_ownership"]:
    print(f"--- {column} ({loans[column].dtype})")
    print(loans[column].value_counts(dropna=False).head(8))
    print()

--- term (str)
term
36 months    1023206
60 months     324893
Name: count, dtype: int64

--- emp_length (str)
emp_length
10+ years    442679
2 years      122100
< 1 year     108537
3 years      107868
1 year        88843
5 years       84326
4 years       80763
NaN           78550
Name: count, dtype: int64

--- int_rate (float64)
int_rate
10.99    37666
11.99    33112
13.99    29360
5.32     28820
9.17     26048
12.99    25590
7.89     25054
15.61    22752
Name: count, dtype: int64

--- revol_util (float64)
revol_util
0.00     7066
57.00    2657
58.00    2640
53.00    2636
54.00    2631
59.00    2609
48.00    2600
61.00    2587
Name: count, dtype: int64

--- issue_d (str)
issue_d
Mar-2016    48938
Oct-2015    42986
Jul-2015    41053
Dec-2015    39003
Oct-2014    36072
Feb-2016    34599
Nov-2015    32908
Jan-2015    32216
Name: count, dtype: int64

--- home_ownership (str)
home_ownership
MORTGAGE    666852
RENT        535699
OWN         145027
ANY            286
OTHER          182
NONE  

## 7. Missing values (selected columns)
- `emp_length` is missing for 5.8% of loans. Missing employment length is likely informative (no job / not reported), so it is kept as a separate **"Unknown"** category instead of being dropped or imputed.
- All other columns have less than 0.1% missing values.

## 8. Type conversions
- `term`: " 36 months" → `term_months` (36 / 60)
- `emp_length`: missing → "Unknown"; numeric version `emp_length_years` ("< 1 year" → 0, "10+ years" → 10)
- `issue_d`: "Dec-2015" → `issue_date`, plus `issue_year` and `issue_quarter` for cohort analysis
- `home_ownership`: rare values ANY / NONE merged into OTHER (521 loans)
- `int_rate`, `revol_util` were already numeric

In [22]:
loans["term_months"] = loans["term"].str.extract(r"(\d+)", expand=False).astype("int16")
loans["term_months"].value_counts()

term_months
36    1023206
60     324893
Name: count, dtype: int64

In [23]:
loans["emp_length"] = loans["emp_length"].fillna("Unknown")

loans["emp_length_years"] = (
    loans["emp_length"].str.extract(r"(\d+)", expand=False).astype("Int64")
)
loans.loc[loans["emp_length"] == "< 1 year", "emp_length_years"] = 0

loans[["emp_length", "emp_length_years"]].value_counts(dropna=False).sort_index()

emp_length  emp_length_years
1 year      1                    88843
10+ years   10                  442679
2 years     2                   122100
3 years     3                   107868
4 years     4                    80763
5 years     5                    84326
6 years     6                    62879
7 years     7                    59724
8 years     8                    60811
9 years     9                    51019
< 1 year    0                   108537
Unknown     <NA>                 78550
Name: count, dtype: int64

In [24]:
loans["home_ownership"] = loans["home_ownership"].replace({"NONE": "OTHER", "ANY": "OTHER"})
loans["home_ownership"].value_counts()

home_ownership
MORTGAGE    666852
RENT        535699
OWN         145027
OTHER          521
Name: count, dtype: int64

In [25]:
loans["issue_date"] = pd.to_datetime(loans["issue_d"], format="%b-%Y", errors="coerce")

print("Unparsed dates:", loans["issue_date"].isna().sum())
print("Range:", loans["issue_date"].min(), "->", loans["issue_date"].max())

Unparsed dates: 0
Range: 2007-06-01 00:00:00 -> 2018-12-01 00:00:00


In [26]:
loans["issue_year"] = loans["issue_date"].dt.year.astype("int16")
loans["issue_quarter"] = loans["issue_date"].dt.to_period("Q").astype(str)

loans[["issue_d", "issue_date", "issue_year", "issue_quarter"]].head()

,issue_d,issue_date,issue_year,issue_quarter
0,Dec-2015,2015-12-01,2015,2015Q4
1,Dec-2015,2015-12-01,2015,2015Q4
2,Dec-2015,2015-12-01,2015,2015Q4
4,Dec-2015,2015-12-01,2015,2015Q4
5,Dec-2015,2015-12-01,2015,2015Q4


## 9. Derived columns
- `fico_avg`: midpoint of the FICO range (mean 698)
- `net_loss`: for defaulted loans `loan_amnt − total_pymnt` (floored at 0), otherwise 0. `total_pymnt` already includes recoveries, so they are not added again.

| Metric | Value |
| --- | --- |
| Total net loss | ~$1.99B |
| Average loss per defaulted loan | ~$7,400 |
| Loss rate on defaulted loans | ~47.6% |

In [27]:
loans["fico_avg"] = (loans["fico_range_low"] + loans["fico_range_high"]) / 2
loans["fico_avg"].describe()

count   1,348,099.00
mean          698.16
std            31.85
min           612.00
25%           672.00
50%           692.00
75%           712.00
max           847.50
Name: fico_avg, dtype: float64

In [28]:
loans["net_loss"] = np.where(
    loans["is_default"] == 1,
    (loans["loan_amnt"] - loans["total_pymnt"]).clip(lower=0),
    0.0,
)

defaults = loans[loans["is_default"] == 1]
print(f"Total net loss:   ${loans['net_loss'].sum():,.0f}")
print(f"Avg loss/default: ${defaults['net_loss'].mean():,.0f}")
print(f"Loss rate on defaulted loans: {defaults['net_loss'].sum() / defaults['loan_amnt'].sum():.1%}")

Total net loss:   $1,994,182,686
Avg loss/default: $7,403
Loss rate on defaulted loans: 47.6%


## 10. Outliers and invalid values
Rule: impossible values are removed or set to missing; extreme but possible values are kept.

| Column | Issue | Rule | Rows affected |
| --- | --- | --- | --- |
| `annual_inc` | 0, missing or < $1,000 | row dropped | 401 |
| `annual_inc` | up to $11M | kept (use median in analysis) | 291 above $1M |
| `dti` | −1 and 999 (placeholders) | set to missing | 535 |
| `revol_util` | up to 892% | values > 150 set to missing | 20 |

In [29]:
numeric_columns = [
    "loan_amnt", "int_rate", "installment", "annual_inc", "dti",
    "revol_util", "open_acc", "delinq_2yrs", "inq_last_6mths",
]
loans[numeric_columns].describe(percentiles=[0.01, 0.5, 0.99]).T

,count,mean,std,min,1%,50%,99%,max
loan_amnt,"1,348,099.00","14,409.00","8,716.14",500.00,"1,500.00","12,000.00","35,000.00","40,000.00"
int_rate,"1,348,099.00",13.24,4.77,5.31,5.32,12.74,26.30,30.99
installment,"1,348,099.00",437.78,261.50,4.93,52.75,375.04,"1,220.33","1,719.83"
annual_inc,"1,348,095.00","76,237.74","69,922.74",0.00,"18,000.00","65,000.00","251,000.00","10,999,200.00"
dti,"1,347,725.00",18.27,11.16,-1.00,1.77,17.61,38.47,999.00
revol_util,"1,347,202.00",51.81,24.53,0.00,1.10,52.20,98.20,892.30
open_acc,"1,348,070.00",11.59,5.47,0.00,3.00,11.00,29.00,90.00
delinq_2yrs,"1,348,070.00",0.32,0.88,0.00,0.00,0.00,4.00,39.00
inq_last_6mths,"1,348,069.00",0.66,0.96,0.00,0.00,0.00,4.00,33.00


In [30]:
print("annual_inc <= 0:        ", (loans["annual_inc"] <= 0).sum())
print("annual_inc missing:     ", loans["annual_inc"].isna().sum())
print("annual_inc > 1M:        ", (loans["annual_inc"] > 1_000_000).sum())
print("dti < 0:                ", (loans["dti"] < 0).sum())
print("dti > 100:              ", (loans["dti"] > 100).sum())
print("revol_util > 100:       ", (loans["revol_util"] > 100).sum())
print("revol_util > 150:       ", (loans["revol_util"] > 150).sum())

annual_inc <= 0:         361
annual_inc missing:      4
annual_inc > 1M:         291
dti < 0:                 2
dti > 100:               533
revol_util > 100:        4714
revol_util > 150:        20


In [31]:
rows_before = len(loans)

# annual_inc: drop rows with zero or missing income
loans = loans[loans["annual_inc"] > 0].copy()

# dti: impossible values -> missing
loans.loc[(loans["dti"] < 0) | (loans["dti"] > 100), "dti"] = np.nan

# revol_util: implausible values -> missing
loans.loc[loans["revol_util"] > 150, "revol_util"] = np.nan

print(f"Rows dropped: {rows_before - len(loans):,}")
print(f"Rows left:    {len(loans):,}")

Rows dropped: 365
Rows left:    1,347,734


In [32]:
loans[["annual_inc", "dti", "revol_util"]].describe(percentiles=[0.01, 0.99]).T

,count,mean,std,min,1%,99%,max
annual_inc,"1,347,734.00","76,258.16","69,920.97",1.00,"18,000.00","251,000.00","10,999,200.00"
dti,"1,347,186.00",18.18,8.68,0.00,1.77,38.36,100.00
revol_util,"1,346,822.00",51.81,24.51,0.00,1.10,98.20,148.00


In [33]:
for limit in [100, 1_000, 5_000, 10_000]:
    print(f"annual_inc < {limit:>6,}: {(loans['annual_inc'] < limit).sum():,}")

annual_inc <    100: 17
annual_inc <  1,000: 36
annual_inc <  5,000: 198
annual_inc < 10,000: 1,187


In [34]:
loans = loans[loans["annual_inc"] >= 1_000].copy()
print(f"Rows left: {len(loans):,}")
print(f"Min income: {loans['annual_inc'].min():,.0f}")

Rows left: 1,347,698
Min income: 1,000


## 11. Final checks
Remaining missing values (`dti`, `revol_util`, a few credit-history fields) are left as missing on purpose. Imputation is a modeling decision and will be done in the model notebook using training data only, to avoid data leakage.

In [35]:
FINAL_COLUMNS = [
    "id",
    "issue_date", "issue_year", "issue_quarter",
    "loan_amnt", "term_months", "int_rate", "installment",
    "grade", "sub_grade", "purpose",
    "emp_length", "emp_length_years", "home_ownership", "annual_inc",
    "verification_status", "addr_state",
    "dti", "fico_range_low", "fico_range_high", "fico_avg",
    "revol_util", "open_acc", "delinq_2yrs", "inq_last_6mths",
    "loan_status", "is_default", "total_pymnt", "recoveries", "net_loss",
]

loans = loans[FINAL_COLUMNS].reset_index(drop=True)
print(loans.shape)

(1347698, 30)


In [36]:
loans.head()

,id,issue_date,issue_year,issue_quarter,loan_amnt,term_months,int_rate,installment,grade,sub_grade,purpose,emp_length,emp_length_years,home_ownership,annual_inc,verification_status,addr_state,dti,fico_range_low,fico_range_high,fico_avg,revol_util,open_acc,delinq_2yrs,inq_last_6mths,loan_status,is_default,total_pymnt,recoveries,net_loss
0,68407277,2015-12-01,2015,2015Q4,"3,600.00",36,13.99,123.03,C,C4,debt_consolidation,10+ years,10,MORTGAGE,"55,000.00",Not Verified,PA,5.91,675.00,679.00,677.00,29.70,7.00,0.00,1.00,Fully Paid,0,"4,421.72",0.00,0.00
1,68355089,2015-12-01,2015,2015Q4,"24,700.00",36,11.99,820.28,C,C1,small_business,10+ years,10,MORTGAGE,"65,000.00",Not Verified,SD,16.06,715.00,719.00,717.00,19.20,22.00,1.00,4.00,Fully Paid,0,"25,679.66",0.00,0.00
2,68341763,2015-12-01,2015,2015Q4,"20,000.00",60,10.78,432.66,B,B4,home_improvement,10+ years,10,MORTGAGE,"63,000.00",Not Verified,IL,10.78,695.00,699.00,697.00,56.20,6.00,0.00,0.00,Fully Paid,0,"22,705.92",0.00,0.00
3,68476807,2015-12-01,2015,2015Q4,"10,400.00",60,22.45,289.91,F,F1,major_purchase,3 years,3,MORTGAGE,"104,433.00",Source Verified,PA,25.37,695.00,699.00,697.00,64.50,12.00,1.00,3.00,Fully Paid,0,"11,740.50",0.00,0.00
4,68426831,2015-12-01,2015,2015Q4,"11,950.00",36,13.44,405.18,C,C3,debt_consolidation,4 years,4,RENT,"34,000.00",Source Verified,GA,10.20,690.00,694.00,692.00,68.40,5.00,0.00,0.00,Fully Paid,0,"13,708.95",0.00,0.00


In [37]:
checks = {
    "unique id":            loans["id"].is_unique,
    "target only 0/1":      set(loans["is_default"].unique()) <= {0, 1},
    "term only 36/60":      set(loans["term_months"].unique()) <= {36, 60},
    "income >= 1,000":      (loans["annual_inc"] >= 1_000).all(),
    "dti within 0-100":     loans["dti"].dropna().between(0, 100).all(),
    "net_loss >= 0":        (loans["net_loss"] >= 0).all(),
    "no loss if paid":      (loans.loc[loans["is_default"] == 0, "net_loss"] == 0).all(),
}
for name, passed in checks.items():
    print(f"{name:<20} {passed}")

unique id            True
target only 0/1      True
term only 36/60      True
income >= 1,000      True
dti within 0-100     True
net_loss >= 0        True
no loss if paid      True


In [38]:
loans.info()

<class 'pandas.DataFrame'>
RangeIndex: 1347698 entries, 0 to 1347697
Data columns (total 30 columns):
 #   Column               Non-Null Count    Dtype         
---  ------               --------------    -----         
 0   id                   1347698 non-null  str           
 1   issue_date           1347698 non-null  datetime64[us]
 2   issue_year           1347698 non-null  int16         
 3   issue_quarter        1347698 non-null  str           
 4   loan_amnt            1347698 non-null  float64       
 5   term_months          1347698 non-null  int16         
 6   int_rate             1347698 non-null  float64       
 7   installment          1347698 non-null  float64       
 8   grade                1347698 non-null  str           
 9   sub_grade            1347698 non-null  str           
 10  purpose              1347698 non-null  str           
 11  emp_length           1347698 non-null  str           
 12  emp_length_years     1269520 non-null  Int64         
 13  home_own

In [39]:
loans.isna().sum()

id                         0
issue_date                 0
issue_year                 0
issue_quarter              0
loan_amnt                  0
term_months                0
int_rate                   0
installment                0
grade                      0
sub_grade                  0
purpose                    0
emp_length                 0
emp_length_years       78178
home_ownership             0
annual_inc                 0
verification_status        0
addr_state                 0
dti                      516
fico_range_low             0
fico_range_high            0
fico_avg                   0
revol_util               912
open_acc                  25
delinq_2yrs               25
inq_last_6mths            26
loan_status                0
is_default                 0
total_pymnt                0
recoveries                 0
net_loss                   0
dtype: int64

In [40]:
loans["id"] = loans["id"].astype("int64")
loans["id"].dtype

dtype('int64')

## 12. Save and load to PostgreSQL
- Saved as **parquet** to keep data types (dates, nullable integers) and for fast reloading
- Table schema is defined explicitly in `sql/01_schema.sql`; data is appended with `to_sql`
- Row count and default rate match between pandas and PostgreSQL

In [41]:
PROCESSED_FILE = Path("../data/processed/loans_clean.parquet")

loans.to_parquet(PROCESSED_FILE, index=False)
print(f"Saved: {PROCESSED_FILE} ({PROCESSED_FILE.stat().st_size / 1e6:.0f} MB)")

check = pd.read_parquet(PROCESSED_FILE)
print(check.shape)
print(check.dtypes)

Saved: ..\data\processed\loans_clean.parquet (46 MB)
(1347698, 30)
id                              int64
issue_date             datetime64[us]
issue_year                      int16
issue_quarter                     str
loan_amnt                     float64
term_months                     int16
int_rate                      float64
installment                   float64
grade                             str
sub_grade                         str
purpose                           str
emp_length                        str
emp_length_years                Int64
home_ownership                    str
annual_inc                    float64
verification_status               str
addr_state                        str
dti                           float64
fico_range_low                float64
fico_range_high               float64
fico_avg                      float64
revol_util                    float64
open_acc                      float64
delinq_2yrs                   float64
inq_last_6mths       

In [44]:
load_dotenv("../.env")
engine = create_engine(os.getenv("DATABASE_URL"))

to_load = loans.copy()
to_load["issue_date"] = to_load["issue_date"].dt.date   # timestamp -> date
to_load.to_sql("loans", engine, if_exists="append", index=False, chunksize=10_000)

print("Loaded.")

Loaded.


In [45]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            COUNT(*)                         AS loan_count,
            ROUND(AVG(is_default) * 100, 2)  AS default_rate_pct,
            SUM(net_loss)                    AS total_net_loss
        FROM loans
    """)).one()

print(result)
print(f"Rows in pandas:         {len(loans):,}")
print(f"Default rate in pandas: {loans['is_default'].mean():.2%}")

(1347698, Decimal('19.98'), Decimal('1993136445.12'))
Rows in pandas:         1,347,698
Default rate in pandas: 19.98%


## 13. Accepted applications per month and state
`loans` contains only completed loans, so it cannot be used to count approvals (most 2016–2018 loans were still open).
This summary counts **all** accepted loans from the raw file, regardless of their status, for approval rate analysis.

In [46]:
accepted = pd.read_csv(
    ACCEPTED_FILE,
    usecols=["id", "issue_d", "addr_state", "loan_amnt"],
    dtype={"id": "string"},
)
print(accepted.shape)
print(accepted.isna().sum())

(2260701, 4)
id             0
loan_amnt     33
issue_d       33
addr_state    33
dtype: int64


In [47]:
accepted = accepted.dropna(subset=["issue_d", "addr_state"])
accepted["month"] = pd.to_datetime(accepted["issue_d"], format="%b-%Y")

accepted_monthly = (
    accepted.groupby(["month", "addr_state"])
    .agg(
        accepted_count=("id", "size"),
        accepted_amount=("loan_amnt", "sum"),
    )
    .reset_index()
)

print(accepted_monthly.shape)
accepted_monthly.head()

(5986, 4)


,month,addr_state,accepted_count,accepted_amount
0,2007-06-01,CO,1,"2,600.00"
1,2007-06-01,CT,4,"28,000.00"
2,2007-06-01,FL,1,"2,000.00"
3,2007-06-01,GA,1,"6,450.00"
4,2007-06-01,KS,1,"3,000.00"


In [48]:
print(f"Total accepted loans: {accepted_monthly['accepted_count'].sum():,}")
print(f"Months: {accepted_monthly['month'].nunique()}")
print(f"States: {accepted_monthly['addr_state'].nunique()}")
print("Range:", accepted_monthly["month"].min(), "->", accepted_monthly["month"].max())

Total accepted loans: 2,260,668
Months: 139
States: 51
Range: 2007-06-01 00:00:00 -> 2018-12-01 00:00:00


In [49]:
ACCEPTED_MONTHLY_FILE = Path("../data/processed/accepted_monthly.parquet")

accepted_monthly.to_parquet(ACCEPTED_MONTHLY_FILE, index=False)
print(f"Saved: {ACCEPTED_MONTHLY_FILE} ({ACCEPTED_MONTHLY_FILE.stat().st_size / 1e3:.0f} KB)")

del accepted

Saved: ..\data\processed\accepted_monthly.parquet (60 KB)


In [51]:
load_dotenv("../.env")
engine = create_engine(os.getenv("DATABASE_URL"))

to_load = pd.read_parquet("../data/processed/accepted_monthly.parquet")
to_load["month"] = to_load["month"].dt.date   # timestamp -> date

to_load.to_sql("accepted_monthly", engine, if_exists="append", index=False)

with engine.connect() as connection:
    db_total = connection.execute(text("SELECT SUM(accepted_count) FROM accepted_monthly")).scalar()

print(f"Rows loaded:    {len(to_load):,}")
print(f"Total in file:  {to_load['accepted_count'].sum():,}")
print(f"Total in DB:    {db_total:,}")

Rows loaded:    5,986
Total in file:  2,260,668
Total in DB:    2,260,668


## Summary

| Step | Rows |
| --- | --- |
| Raw data | 2,260,701 |
| Completed loans | 1,348,099 |
| Income 0 or missing removed | 1,347,734 |
| Income < $1,000 removed | **1,347,698** |

**Final table:** 1,347,698 loans · default rate 19.98% · total net loss ~$1.99B

**Note for analysis:** loans issued in 2017–2018 are over-represented by early payoffs and early defaults, because most of them had not reached maturity when the data ends. This selection bias must be considered in yearly and vintage analysis.

In [50]:
raw_status = pd.read_csv(ACCEPTED_FILE, usecols=["issue_d", "term", "loan_status"])
raw_status = raw_status.dropna(subset=["issue_d"])

raw_status["issue_year"] = pd.to_datetime(raw_status["issue_d"], format="%b-%Y").dt.year
raw_status["term_months"] = raw_status["term"].str.extract(r"(\d+)", expand=False).astype(int)
raw_status["is_open"] = ~raw_status["loan_status"].isin(STATUS_TO_DEFAULT.keys())

open_share = (
    raw_status.pivot_table(index="issue_year", columns="term_months", values="is_open", aggfunc="mean")
    * 100
).round(1)
open_share

term_months,36,60
issue_year,,
2007,0.00,NaN
2008,0.00,NaN
2009,0.00,NaN
2010,0.00,0.00
2011,0.00,0.00
2012,0.00,0.00
2013,0.00,0.00
2014,0.00,17.10
2015,0.10,32.90


### Maturity check
Share of loans still open (Current / Late / In Grace Period) by issue year and term, in the raw data.
36-month loans are fully resolved up to 2015 and 60-month loans up to 2013.
These thresholds define the `loans_mature` view used for trend and segment analysis.